# Imports

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

# Konstante in parametri rakete (enako kot na vaji 02)
G, M_Z, R_Z = 6.67430e-11, 5.972e24, 6371e3
rho_0, M_zraka, R_plin, T_0 = 1.225, 28.96, 8314.46, 288.15
C_d, S = 0.2, 10.0
F_pot, v_izp = 845e3, 2500.0
mdot = F_pot / v_izp
m_rak, m_tov = 10_000, 1_000
g0 = G * M_Z / R_Z**2
H = R_plin * T_0 / (M_zraka * g0)

# Funkcije

In [ ]:
def gostota(h):
    return rho_0 * np.exp(-np.maximum(h, 0) / H)


def pospesek(h, v, m_g, m_t=m_tov, delez=1.0):
    """Pospešek rakete (potisk + gravitacija + upor) - model z vaje 02.
    delez: del koraka, v katerem motor še deluje (zadnji korak pred izgorevanjem)."""
    m = m_rak + m_t + m_g
    F = (delez * F_pot if m_g > 0 else 0.0) - G * M_Z * m / (R_Z + h)**2 \
        - 0.5 * gostota(h) * C_d * S * v * abs(v)
    return F / m


def simulacija_euler(m_g0, dt, m_t=m_tov):
    """Eksplicitna Eulerjeva metoda do najvišje točke. Vrne DataFrame (t, h, v, m_g)."""
    t, h, v, m_g = 0.0, 0.0, 0.0, float(m_g0)
    zapis = [(t, h, v, m_g)]
    while True:
        delez = min(1.0, m_g / (mdot * dt))   # gorivo lahko pošle sredi koraka
        a = pospesek(h, v, m_g, m_t, delez)
        h, v, m_g, t = h + v * dt, v + a * dt, max(m_g - mdot * dt, 0.0), t + dt
        zapis.append((t, h, v, m_g))
        if v < 0:
            break
    return pd.DataFrame(zapis, columns=["t", "h", "v", "m_g"])


def desna_stran(t, y, m_t=m_tov):
    """Desna stran sistema dy/dt za y = [h, v, m_g] (za solve_ivp)."""
    h, v, m_g = y
    # TODO: vrni [dh/dt, dv/dt, dm_g/dt]; poraba goriva je mdot, dokler m_g > 0
    pass


def dogodek_vrh(t, y, m_t=m_tov):
    return y[1]            # v = 0
dogodek_vrh.terminal = True
dogodek_vrh.direction = -1


def simulacija_ivp(m_g0, m_t=m_tov, rtol=1e-8):
    """Let v dveh delih: do izgorevanja (t_b = m_g0/mdot) in od izgorevanja do vrha.
    Vrne DataFrame (t, h, v, m_g) na gosti mreži časov."""
    t_b = m_g0 / mdot
    # TODO: 1. del: solve_ivp od 0 do t_b (dense_output=True)
    # TODO: 2. del: od t_b do vrha z dogodkom dogodek_vrh
    # TODO: združi oba dela v DataFrame
    pass


def analiza_leta(df):
    """Vrne slovar: h_max, t_vrh, a_max, q_max (dinamični tlak) in h_q (višina pri q_max)."""
    a = np.gradient(df["v"].to_numpy(), df["t"].to_numpy())
    # TODO: q = rho v^2 / 2
    pass

# Testno območje

In [ ]:
# P1: prosti met navzgor z Eulerjevo metodo (g = 10 m/s^2, v0 = 50 m/s, dt = 1 s)
h, v, dt = 0.0, 50.0, 1.0
for n in range(3):
    h, v = h + v * dt, v - 10 * dt
print("Euler:   h(3 s) =", h, " v(3 s) =", v)
print("točno:   h(3 s) =", 50 * 3 - 5 * 3**2, " v(3 s) =", 50 - 10 * 3)

# P3: dinamični tlak pri konstantnem pospešku ima vrh pri h = H
a0 = (F_pot - (m_rak + m_tov + 20_000) * g0) / (m_rak + m_tov + 20_000)
print(f"a0 = {a0:.2f} m/s^2, H = {H:.0f} m, q_max(ocena) = {rho_0 * a0 * H / np.e / 1e3:.1f} kPa")

# Naloga 1

In [ ]:
m_g0 = 20_000
koraki = [10, 5, 2, 1, 0.1, 0.01]
t0 = time.time()
ref = analiza_leta(simulacija_euler(m_g0, 0.001))  # referenčna rešitev
print(f"referenca (dt = 0.001 s): h_max = {ref['h_max']/1e3:.3f} km, {time.time()-t0:.1f} s")
rez = []  # TODO: za vsak dt h_max, t_vrh, napaka |h_max - h_ref| in čas izvajanja (Preglednica 1)

# Naloga 1 - Slika 1

In [ ]:
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(121)
# TODO: h(t) za dt = 10, 1 in 0.01 s
ax = fig.add_subplot(122)
# TODO: napaka v odvisnosti od dt (log-log) in naklon premice (np.polyfit)
fig.tight_layout()
plt.show()

# Naloga 2

In [ ]:
# TODO: simulacija_ivp za m_g0 = 20 t; primerjava z Eulerjem (Preglednica 2)

# Naloga 3

In [ ]:
mase = [1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000]
studija = []  # TODO: analiza_leta(simulacija_ivp(m)) za vse mase goriva (Preglednica 3)

# Naloga 3 - Slika 2

In [ ]:
fig = plt.figure(figsize=(11, 4))
ax = fig.add_subplot(121)
# TODO: h_max v odvisnosti od m_g (log os x)
ax = fig.add_subplot(122)
# TODO: dinamični tlak q(h) za vse mase goriva (do 60 km)
fig.tight_layout()
plt.show()

# Naloga 4

In [ ]:
# TODO: brentq za najmanjšo maso goriva, pri kateri h_max = 100 km (Preglednica 4)